In [1]:
# Imports and load .env
import os
import time
import requests
import json
import pandas as pd
from dotenv import load_dotenv

load_dotenv()  # load BRIGHTDATA_API_KEY from .env

api_token = os.getenv("BRIGHTDATA_API_KEY")
dataset_id = "gd_l7j1po0921hbu0ri1z"
company_url = "https://www.glassdoor.com/Overview/Working-at-DaVita-EI_IE1432.11,17.htm"

assert api_token, "API key not found! Please check your .env file."


In [2]:
# Define function to trigger dataset
def trigger_dataset(api_token, dataset_id, company_url, days=10000):
    headers = {
        "Authorization": f"Bearer {api_token}",
        "Content-Type": "application/json",
    }
    payload = json.dumps([
        {
            "url": company_url,
            "days": days
        }
    ])
    response = requests.post(
        "https://api.brightdata.com/datasets/v3/trigger",
        headers=headers,
        params={"dataset_id": dataset_id},
        data=payload,
    )
    return response.json()


In [3]:
# Check snapshot status
def check_snapshot_status(api_token, snapshot_id):
    url = f"https://api.brightdata.com/datasets/v3/snapshot/{snapshot_id}"
    headers = {"Authorization": f"Bearer {api_token}"}
    response = requests.get(url, headers=headers)
    return response.json()


In [4]:
# Trigger scraping job
response_data = trigger_dataset(api_token, dataset_id, company_url)
print("Trigger response:")
print(json.dumps(response_data, indent=2))

snapshot_id = response_data.get("snapshot_id")
if not snapshot_id:
    raise ValueError("No snapshot_id returned. Check your dataset ID and input.")
print(f"Snapshot ID: {snapshot_id}")


Trigger response:
{
  "snapshot_id": "s_mdvxbfh21ba5nx8mea"
}
Snapshot ID: s_mdvxbfh21ba5nx8mea


In [5]:
#  Poll snapshot status until ready
status = None
max_tries = 20
tries = 0

while status != "completed" and tries < max_tries:
    snapshot_info = check_snapshot_status(api_token, snapshot_id)
    status = snapshot_info.get("status")
    print(f"Check {tries+1}: Snapshot status = {status}")
    if status == "completed":
        break
    elif status == "failed":
        raise RuntimeError("Snapshot failed to generate.")
    tries += 1
    time.sleep(30)

if status != "completed":
    raise TimeoutError(f"Snapshot not ready after {max_tries*30} seconds.")


Check 1: Snapshot status = running


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
# Download snapshot data and save to CSV
data_url = f"https://api.brightdata.com/datasets/v3/snapshot/{snapshot_id}?format=json"
headers = {"Authorization": f"Bearer {api_token}"}
snapshot_response = requests.get(data_url, headers=headers)
data = snapshot_response.json()

# Check if data is a list or nested, adjust as needed
if isinstance(data, list):
    df = pd.DataFrame(data)
elif 'result' in data:
    df = pd.DataFrame(data['result'])
else:
    df = pd.DataFrame([data])

print(f"Downloaded {len(df)} records.")

csv_filename = "davita_reviews.csv"
df.to_csv(csv_filename, index=False)
print(f"Saved data to {csv_filename}")
